# Apache Iceberg file pruning

## What is file pruning?
- 쿼리와 상관 없는 불필요한 파일을 스캔하지 않고 건너뛰는 것을 의미한다.
- 예를 들어, year 로 파티셔닝 되어 있는 데이터가 2020년부터 2025년까지 있을 때 아래와 같이 쿼리하면 2020~2024년 데이터 파일은 스캔하지 않고 2025년 데이터만 읽는 것이다.
```
SELECT * FROM table WHERE year=2025;
```

## Hive vs Iceberg 의 file pruning은 어떤 차이점이 있나?
- Hive는 테이블 파티션 기준으로 파일 프루닝을 지원한다.
- Iceberg는 파티션되지 않은 컬럼도 필터를 적용하여 파일 프루닝을 수행할 수 있다. 
  - 이유는 manifest file 에서 참조 중인 각 data file(parquet)의 컬럼 통계 정보를 갖고 있기 때문이다.

In [54]:
import io
import json
import boto3
from fastavro import reader as avro_reader
from pyspark.sql import SparkSession
from utils.iceberg_metadata_properties import IcebergMetadataJsonField, FIELD_DESCRIPTIONS
from utils.get_iceberg_metadata import get_metadata
from pyspark.sql.functions import input_file_name

In [66]:
spark = SparkSession.builder \
    .appName("IcebergSparkWithMinIO") \
    .config("spark.sql.catalog.my_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.my_catalog.type", "rest") \
    .config("spark.sql.catalog.my_catalog.uri", "http://iceberg-rest:8181") \
    .config("spark.sql.catalog.my_catalog.warehouse", "s3://warehouse/") \
    .config("spark.sql.catalog.my_catalog.io-impl", "org.apache.iceberg.aws.s3.S3FileIO") \
    .config("spark.sql.catalog.my_catalog.s3.endpoint", "http://minio:9000") \
    .config("spark.sql.catalog.my_catalog.s3.access-key-id", "admin") \
    .config("spark.sql.catalog.my_catalog.s3.secret-access-key", "password") \
    .config("spark.sql.catalog.my_catalog.s3.path-style-access", "true") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.defaultCatalog", "my_catalog") \
    .getOrCreate()

# Verify setup
spark.sql("SHOW CATALOGS").show()

+-------------+
|      catalog|
+-------------+
|   my_catalog|
|spark_catalog|
+-------------+



In [40]:
spark.sql("""
CREATE TABLE IF NOT EXISTS my_catalog.my_database.sales_v2 (
    id BIGINT,
    amount DOUBLE,
    region STRING,
    date STRING
)
USING iceberg
PARTITIONED BY (date);
""")

DataFrame[]

In [41]:
spark.sql("""SHOW TABLES IN my_database""").show()

+-----------+---------+-----------+
|  namespace|tableName|isTemporary|
+-----------+---------+-----------+
|my_database|    sales|      false|
|my_database| sales_v2|      false|
+-----------+---------+-----------+



In [5]:
spark.sql("""
INSERT INTO my_catalog.my_database.sales_v2 VALUES
(1, 100.5, 'North', '2024-01-01'),
(2, 200.0, 'South', '2024-01-01'),
(3, 150.0, 'North', '2024-01-02'),
(4, 250.0, 'West', '2024-01-02'),
(5, 300.0, 'East', '2024-01-03');
""")

DataFrame[]

In [14]:
spark.sql("""
SELECT * FROM my_catalog.my_database.sales_v2 ORDER BY date
""").show()

+---+------+------+----------+
| id|amount|region|      date|
+---+------+------+----------+
|  1| 100.5| North|2024-01-01|
|  2| 200.0| South|2024-01-01|
|  3| 150.0| North|2024-01-02|
|  4| 250.0|  West|2024-01-02|
|  5| 300.0|  East|2024-01-03|
+---+------+------+----------+



## Parition 확인
- 아래와 같이 date 컬럼을 기준으로 data file 이 3개로 파티션 되어 있다.

In [11]:
spark.sql("""
    SELECT file_path,partition 
    FROM my_catalog.my_database.sales_v2.files 
    ORDER BY partition
""").show(truncate=False)

+---------------------------------------------------------------------------------------------------------------------+------------+
|file_path                                                                                                            |partition   |
+---------------------------------------------------------------------------------------------------------------------+------------+
|s3://warehouse/my_database/sales_v2/data/date=2024-01-01/00000-5-1da21270-2476-43ce-80eb-2bea0cef1f5a-0-00001.parquet|{2024-01-01}|
|s3://warehouse/my_database/sales_v2/data/date=2024-01-02/00000-5-1da21270-2476-43ce-80eb-2bea0cef1f5a-0-00002.parquet|{2024-01-02}|
|s3://warehouse/my_database/sales_v2/data/date=2024-01-03/00000-5-1da21270-2476-43ce-80eb-2bea0cef1f5a-0-00003.parquet|{2024-01-03}|
+---------------------------------------------------------------------------------------------------------------------+------------+



- 현재 2024-01-02 파티션에는 하나의 데이터 파일만 존재한다.

In [18]:
spark.sql("""
    SELECT file_path, partition 
    FROM my_catalog.my_database.sales_v2.files 
    WHERE partition.date = '2024-01-02'
""").show(truncate=False)

+---------------------------------------------------------------------------------------------------------------------+------------+
|file_path                                                                                                            |partition   |
+---------------------------------------------------------------------------------------------------------------------+------------+
|s3://warehouse/my_database/sales_v2/data/date=2024-01-02/00000-5-1da21270-2476-43ce-80eb-2bea0cef1f5a-0-00002.parquet|{2024-01-02}|
+---------------------------------------------------------------------------------------------------------------------+------------+



- 그리고 2024-01-02 에는 2개의 행이 있다.

In [20]:
spark.sql("""
    SELECT *
    FROM my_catalog.my_database.sales_v2
    WHERE date = '2024-01-02'
    ORDER BY id
""").show(truncate=False)

+---+------+------+----------+
|id |amount|region|date      |
+---+------+------+----------+
|3  |150.0 |North |2024-01-02|
|4  |250.0 |West  |2024-01-02|
+---+------+------+----------+



- 이번에는 spark log level 을 DEBUG 로 설정해서 실제로 `WHERE date = '2024-01-02'` 와 같이 필터링 했을 때 다른 데이터 파일은 읽지 않는지 확인해보자.

In [24]:
spark.sparkContext.setLogLevel("DEBUG")  # To show detailed file access logs

In [25]:
spark.sql("""
    SELECT *
    FROM my_catalog.my_database.sales_v2
    WHERE date = '2024-01-02'
    ORDER BY id
""").show(truncate=False)

+---+------+------+----------+
|id |amount|region|date      |
+---+------+------+----------+
|3  |150.0 |North |2024-01-02|
|4  |250.0 |West  |2024-01-02|
+---+------+------+----------+



25/10/10 13:40:19 DEBUG SparkSqlParser: Parsing command: 
    SELECT *
    FROM my_catalog.my_database.sales_v2
    WHERE date = '2024-01-02'
    ORDER BY id

25/10/10 13:40:19 DEBUG CachingCatalog: Evicted my_database.sales_v2 from the table cache (EXPIRED)
25/10/10 13:40:19 DEBUG InternalHttpClient: ex-0000000030 preparing request execution
25/10/10 13:40:19 DEBUG ProtocolExec: ex-0000000030 target auth state: UNCHALLENGED
25/10/10 13:40:19 DEBUG ProtocolExec: ex-0000000030 proxy auth state: UNCHALLENGED
25/10/10 13:40:19 DEBUG ConnectExec: ex-0000000030 acquiring connection with route {}->[http://iceberg-rest:8181]
25/10/10 13:40:19 DEBUG InternalHttpClient: ex-0000000030 acquiring endpoint (3 MINUTES)
25/10/10 13:40:19 DEBUG PoolingHttpClientConnectionManager: ex-0000000030 endpoint lease request (3 MINUTES) [route: {}->[http://iceberg-rest:8181]][total available: 1; route allocated: 1 of 100; total allocated: 1 of 100]
25/10/10 13:40:19 DEBUG DefaultManagedHttpClientConnection: ht

- log level을 DEBUG로 설정한 후 `WHERE date = '2024-01-02'`와 같이 필터링 했을 때 아래 로그와 같이 `date=2024-01-02` 파티션에 있는 데이터 파일만 읽기 시도하여 data file pruning 이 잘 진행 된다는 것을 알 수 있다. 여기까지는 hive 와 iceberg 모두 동일하다.

- 로그 샘플:
`25/10/10 13:40:20 DEBUG S3InputStream: Seek with new stream for s3://warehouse/my_database/sales_v2/data/date=2024-01-02/00000-5-1da21270-2476-43ce-80eb-2bea0cef1f5a-0-00002.parquet to offset 4`                                                    

### 어떻게, Iceberg는 파티션되지 않은 컬럼도 필터를 적용하면 파일 프루닝을 수행할 수 있을까?

- 테스트를 위해 date가 `2024-01-02` 인 행을 2개 더 추가해보자.

In [32]:
spark.sparkContext.setLogLevel("WARN")  # 불필요한 로그를 출력하지 않기 위해 다시 log level 조정
spark.sql("""INSERT INTO my_catalog.my_database.sales_v2 VALUES 
(6, 250.0, 'South', '2024-01-02'),
(7, 350.0, 'East', '2024-01-02')
""")
spark.sql("SELECT * FROM my_catalog.my_database.sales_v2 ORDER BY id").show()

+---+------+------+----------+
| id|amount|region|      date|
+---+------+------+----------+
|  1| 100.5| North|2024-01-01|
|  2| 200.0| South|2024-01-01|
|  3| 150.0| North|2024-01-02|
|  4| 250.0|  West|2024-01-02|
|  5| 300.0|  East|2024-01-03|
|  6| 250.0| South|2024-01-02|
|  7| 350.0|  East|2024-01-02|
+---+------+------+----------+



- 이제 `date='2024-01-02'` 4개의 행이 되었고 parquet 파일도 2개가 되었다.

In [34]:
spark.sql("SELECT * FROM my_catalog.my_database.sales_v2 WHERE date='2024-01-02'").show()

+---+------+------+----------+
| id|amount|region|      date|
+---+------+------+----------+
|  6| 250.0| South|2024-01-02|
|  7| 350.0|  East|2024-01-02|
|  3| 150.0| North|2024-01-02|
|  4| 250.0|  West|2024-01-02|
+---+------+------+----------+



In [35]:
spark.sql("""
    SELECT file_path, partition 
    FROM my_catalog.my_database.sales_v2.files 
    WHERE partition.date = '2024-01-02'
""").show(truncate=False)

+----------------------------------------------------------------------------------------------------------------------+------------+
|file_path                                                                                                             |partition   |
+----------------------------------------------------------------------------------------------------------------------+------------+
|s3://warehouse/my_database/sales_v2/data/date=2024-01-02/00000-19-83714fd5-ed94-48a2-bfef-99e6f7bbe835-0-00001.parquet|{2024-01-02}|
|s3://warehouse/my_database/sales_v2/data/date=2024-01-02/00000-5-1da21270-2476-43ce-80eb-2bea0cef1f5a-0-00002.parquet |{2024-01-02}|
+----------------------------------------------------------------------------------------------------------------------+------------+



- 이제 WHERE 조건에 date 와 region 을 모두 사용하여 2024-01-02 이면서 East 인 행을 필터링할 때 하나의 데이터 파일(Parquet)만 읽는지 테스트해보자.

In [43]:
spark.sparkContext.setLogLevel("DEBUG") 
spark.sql("""
    SELECT *
    FROM my_catalog.my_database.sales_v2
    WHERE date = '2024-01-02'
    AND region = 'East'
""").show(truncate=False)

+---+------+------+----------+
|id |amount|region|date      |
+---+------+------+----------+
|7  |350.0 |East  |2024-01-02|
+---+------+------+----------+



25/10/10 14:00:41 DEBUG SparkSqlParser: Parsing command: 
    SELECT *
    FROM my_catalog.my_database.sales_v2
    WHERE date = '2024-01-02'
    AND region = 'East'

25/10/10 14:00:41 DEBUG CachingCatalog: Evicted my_database.sales_v2 from the table cache (EXPIRED)
25/10/10 14:00:41 DEBUG InternalHttpClient: ex-0000000073 preparing request execution
25/10/10 14:00:41 DEBUG ProtocolExec: ex-0000000073 target auth state: UNCHALLENGED
25/10/10 14:00:41 DEBUG ProtocolExec: ex-0000000073 proxy auth state: UNCHALLENGED
25/10/10 14:00:41 DEBUG ConnectExec: ex-0000000073 acquiring connection with route {}->[http://iceberg-rest:8181]
25/10/10 14:00:41 DEBUG InternalHttpClient: ex-0000000073 acquiring endpoint (3 MINUTES)
25/10/10 14:00:41 DEBUG PoolingHttpClientConnectionManager: ex-0000000073 endpoint lease request (3 MINUTES) [route: {}->[http://iceberg-rest:8181]][total available: 1; route allocated: 1 of 100; total allocated: 1 of 100]
25/10/10 14:00:41 DEBUG DefaultManagedHttpClientConnec

- 아래 로그와 같이 데이터 파일 2개 중 East 가 있는 데이터 파일 한 개만 읽기 시도 한 것을 알 수 있다.
- 로그 샘플: `25/10/10 13:52:08 DEBUG S3InputStream: Seek with new stream for s3://warehouse/my_database/sales_v2/data/date=2024-01-02/00000-19-83714fd5-ed94-48a2-bfef-99e6f7bbe835-0-00001.parquet to offset 4`
- 어떻게 가능했을까?
- 먼저 메타데이터 포인터에서 바라보는 해당 테이블의 매니패스트 json 파일을 확인해보자.

In [67]:
! curl -s http://iceberg-rest:8181/v1/namespaces/my_database/tables/sales_v2

{"metadata-location":"s3://warehouse/my_database/sales_v2/metadata/00005-88303606-6dac-48ca-8922-b6f19533e201.metadata.json","metadata":{"format-version":2,"table-uuid":"668fc983-4d71-43c4-8b5a-76bd02422423","location":"s3://warehouse/my_database/sales_v2","last-sequence-number":5,"last-updated-ms":1760104001376,"last-column-id":4,"current-schema-id":0,"schemas":[{"type":"struct","schema-id":0,"fields":[{"id":1,"name":"id","required":false,"type":"long"},{"id":2,"name":"amount","required":false,"type":"double"},{"id":3,"name":"region","required":false,"type":"string"},{"id":4,"name":"date","required":false,"type":"string"}]}],"default-spec-id":0,"partition-specs":[{"spec-id":0,"fields":[{"name":"date","transform":"identity","source-id":4,"field-id":1000}]}],"last-partition-id":1000,"default-sort-order-id":0,"sort-orders":[{"order-id":0,"fields":[]}],"properties":{"owner":"root","write.parquet.compression-codec":"zstd"},"current-snapshot-id":4004482107496687407,"refs":{"main":{"snapshot

- 아래와 같이 `00005-88303606-6dac-48ca-8922-b6f19533e201.metadata.json` 내용을 보면 `currnet-snapshot-id` 가 `4004482107496687407` 이다.

In [69]:
metadata_json = "00005-88303606-6dac-48ca-8922-b6f19533e201.metadata.json"
obj = get_metadata(metadata_json, table='sales_v2')
metadata_json = json.loads(obj["Body"].read().decode("utf-8"))

for key in IcebergMetadataJsonField:
    print(f"🔑'KEY':{key.value}({FIELD_DESCRIPTIONS[key]})\n ▶︎'VALUE':{metadata_json[key.value]}")
    print("-"*100)

print("\n📋 메타데이터 원본 JSON")
print(json.dumps(metadata_json, indent=2, ensure_ascii=False))

🔑'KEY':format-version(Iceberg 테이블 포맷 버전(https://iceberg.apache.org/spec/))
 ▶︎'VALUE':2
----------------------------------------------------------------------------------------------------
🔑'KEY':table-uuid(테이블을 고유하게 식별하는 UUID(테이블 변경, 데이터 갱신에도 변경되지 않는 정보))
 ▶︎'VALUE':668fc983-4d71-43c4-8b5a-76bd02422423
----------------------------------------------------------------------------------------------------
🔑'KEY':location(테이블 데이터와 메타데이터가 저장되는 경로)
 ▶︎'VALUE':s3://warehouse/my_database/sales_v2
----------------------------------------------------------------------------------------------------
🔑'KEY':last-sequence-number(마지막으로 커밋된 작업의 시퀀스 번호 (트랜잭션 순서, 테이블 또는 데이터 갱신 작업마다 늘어남))
 ▶︎'VALUE':5
----------------------------------------------------------------------------------------------------
🔑'KEY':last-updated-ms(메타데이터가 마지막으로 갱신된 시각 (epoch 밀리초))
 ▶︎'VALUE':1760104001376
----------------------------------------------------------------------------------------------------
🔑'KEY':last-column-id(테이블

- 이번에는 `currnet-snapshot-id` 가 `4004482107496687407` 인 manifest list(`snap-4004482107496687407-1-0520db04-488c-460a-8198-6b6611e427d7.avro`) 을 확인해보자.
- 아래 출력 결과와 같이 manifest list 는 `0520db04-488c-460a-8198-6b6611e427d7-m0.avro` 와 `50873ac2-c8b8-4d32-9d9f-a9b29118d7d1-m0.avro` 두 개의 manifest file 을 참조하고 있다.

In [70]:
manifest_file = "snap-4004482107496687407-1-0520db04-488c-460a-8198-6b6611e427d7.avro"
obj = get_metadata(file_name=manifest_file, table='sales_v2')
bio = io.BytesIO(obj["Body"].read())
avro_rdr = avro_reader(bio)
for data_file in avro_rdr:
    print(data_file)

{'manifest_path': 's3://warehouse/my_database/sales_v2/metadata/0520db04-488c-460a-8198-6b6611e427d7-m0.avro', 'manifest_length': 7397, 'partition_spec_id': 0, 'content': 0, 'sequence_number': 5, 'min_sequence_number': 5, 'added_snapshot_id': 4004482107496687407, 'added_files_count': 1, 'existing_files_count': 0, 'deleted_files_count': 0, 'added_rows_count': 2, 'existing_rows_count': 0, 'deleted_rows_count': 0, 'partitions': [{'contains_null': False, 'contains_nan': False, 'lower_bound': b'2024-01-02', 'upper_bound': b'2024-01-02'}], 'key_metadata': None}
{'manifest_path': 's3://warehouse/my_database/sales_v2/metadata/50873ac2-c8b8-4d32-9d9f-a9b29118d7d1-m0.avro', 'manifest_length': 7492, 'partition_spec_id': 0, 'content': 0, 'sequence_number': 1, 'min_sequence_number': 1, 'added_snapshot_id': 8219255514662944528, 'added_files_count': 3, 'existing_files_count': 0, 'deleted_files_count': 0, 'added_rows_count': 5, 'existing_rows_count': 0, 'deleted_rows_count': 0, 'partitions': [{'contai

- 이제 `0520db04-488c-460a-8198-6b6611e427d7-m0.avro` 와 `50873ac2-c8b8-4d32-9d9f-a9b29118d7d1-m0.avro` 두 개의 manifest file 내용을 확인해보면 각 data file(parquet) 경로와 함께 data file의 통계정보가 있는 것을 알 수 있다.
- 출력 결과 중 `00000-19-83714fd5-ed94-48a2-bfef-99e6f7bbe835-0-00001.parquet` data file(parquet)의 통계정보에서 lower_bounds 정보중 key 3번(region 컬럼)의 값이 `East` 인 것을 볼 수 있다. 다른 데이터 파일에는 East 가 없다.
- 따라서, Iceberg 는 해당 data file에 Region 이 East ~ South구간만 있다고 판단하고, 파티션 키가 아닌 region 컬럼을 필터링 해도 파티션 프루닝이 지원 된 것이다.
- 이러한 부분이 hive file(parition) pruning 과의 차이점이다.

In [77]:
manifest_file = "0520db04-488c-460a-8198-6b6611e427d7-m0.avro"
obj = get_metadata(file_name=manifest_file, table='sales_v2')
bio = io.BytesIO(obj["Body"].read())
avro_rdr = avro_reader(bio)
for data_file in avro_rdr:
    print(data_file)

manifest_file = "50873ac2-c8b8-4d32-9d9f-a9b29118d7d1-m0.avro"
obj = get_metadata(file_name=manifest_file, table='sales_v2')
bio = io.BytesIO(obj["Body"].read())
avro_rdr = avro_reader(bio)
for data_file in avro_rdr:
    print(data_file)

{'status': 1, 'snapshot_id': 4004482107496687407, 'sequence_number': None, 'file_sequence_number': None, 'data_file': {'content': 0, 'file_path': 's3://warehouse/my_database/sales_v2/data/date=2024-01-02/00000-19-83714fd5-ed94-48a2-bfef-99e6f7bbe835-0-00001.parquet', 'file_format': 'PARQUET', 'partition': {'date': '2024-01-02'}, 'record_count': 2, 'file_size_in_bytes': 1311, 'column_sizes': [{'key': 1, 'value': 51}, {'key': 2, 'value': 51}, {'key': 3, 'value': 52}, {'key': 4, 'value': 82}], 'value_counts': [{'key': 1, 'value': 2}, {'key': 2, 'value': 2}, {'key': 3, 'value': 2}, {'key': 4, 'value': 2}], 'null_value_counts': [{'key': 1, 'value': 0}, {'key': 2, 'value': 0}, {'key': 3, 'value': 0}, {'key': 4, 'value': 0}], 'nan_value_counts': [{'key': 2, 'value': 0}], 'lower_bounds': [{'key': 1, 'value': b'\x06\x00\x00\x00\x00\x00\x00\x00'}, {'key': 2, 'value': b'\x00\x00\x00\x00\x00@o@'}, {'key': 3, 'value': b'East'}, {'key': 4, 'value': b'2024-01-02'}], 'upper_bounds': [{'key': 1, 'value